# Efficiency on the Dalitz Plot with Simulation-Based Inference

This notebook extends `efficiency_dp.ipynb` by building a full SBI pipeline to **infer the
efficiency function parameters** from a reconstructed data sample.

The strategy mirrors what we would do at LHCb:
1. We have a *generator-level* (flat phase-space) MC sample.
2. We have a *reconstruction-level* sample whose distribution is shaped by the detector efficiency.
3. We want to learn the efficiency parameters without building an explicit likelihood.

We use **Neural Posterior Estimation (NPE)** from the `sbi` package.  The notebook covers:
- **Part 1** — Physics setup (Dalitz phase space, polynomial efficiency in $m^2_{ac}$).
- **Part 2** — SBI with a **1-D quadratic efficiency** (3 parameters).
- **Part 3** — Diagnostics: posterior corner plot, posterior-predictive check.
- **Part 4** — Extension to a **2-D efficiency** that depends on both $m^2_{ab}$ and $m^2_{bc}$
  (more realistic for a real detector).

## 0. Imports and phase-space definition

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from sbitest.dalitz import DalitzPhaseSpace
from sbitest.plotting import plot_dalitz

plt.rcParams.update({'figure.dpi': 120})
_ = torch.manual_seed(42)

In [ ]:
# D -> A B C decay kinematics
# ma=0.5 (kaon-like), mb=mc=0.2 (pion-like), md=2.0 (D-meson-like)
phsp = DalitzPhaseSpace(ma=0.5, mb=0.2, mc=0.2, md=2.0)

print(f"Phase-space ranges:")
print(f"  m²_ab : [{phsp.minab:.3f}, {phsp.maxab:.3f}]")
print(f"  m²_bc : [{phsp.minbc:.3f}, {phsp.maxbc:.3f}]")
print(f"  m²_ac : [{phsp.minac:.3f}, {phsp.maxac:.3f}]")

---
## Part 1 — 1-D quadratic efficiency in $m^2_{ac}$

We model the efficiency as a quadratic polynomial in $m^2_{ac}$:

$$\varepsilon(m^2_{ac};\, p_0, p_1, p_2) = \mathrm{clip}\!\left(p_0 + p_1\, m^2_{ac} + p_2\, (m^2_{ac})^2,\; 0, 1\right)$$

The **true** parameter values are taken from `efficiency_dp.ipynb`: a downward-opening
parabola with vertex at $m^2_{ac,\max}$, so $\varepsilon$ rises from $\sim 0.64$ at
$m^2_{ac,\min}$ to $1.0$ at $m^2_{ac,\max}$ — mimicking a selection that preferentially
accepts events in the high-mass corner of the Dalitz plot.

In [ ]:
def efficiency_1d(x, p0, p1, p2):
    """Quadratic efficiency in m2ac.  Returns values in [0,1] after clipping."""
    return np.clip(p0 + p1 * x + p2 * x * x, 0.0, 1.0)

# True parameters matching efficiency_dp.ipynb:
#   p0 = 0.5   (efficiency at x = 0)
#   vertex at x = maxac  →  p1 = -2*p2*maxac,  p2 = -p0 / maxac^2
# This gives eff(minac) ≈ 0.64, eff(maxac) = 1.0  (rising, concave-down parabola)
p0_true = 0.5
p2_true = -p0_true / phsp.maxac ** 2   # negative → concave down
p1_true = -2.0 * p2_true * phsp.maxac  # positive → increasing slope

TRUE_PARAMS = np.array([p0_true, p1_true, p2_true])
print(f"True parameters: p0={p0_true:.4f}, p1={p1_true:.4f}, p2={p2_true:.5f}")

# Verify boundary values
print(f"eff(minac={phsp.minac:.3f}) = {efficiency_1d(phsp.minac, *TRUE_PARAMS):.3f}  (≈0.64)")
print(f"eff(maxac={phsp.maxac:.3f}) = {efficiency_1d(phsp.maxac, *TRUE_PARAMS):.3f}  (= 1.0 by construction)")

In [ ]:
X = np.linspace(phsp.minac, phsp.maxac, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(X, efficiency_1d(X, *TRUE_PARAMS), lw=2, color='steelblue')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel(r"$m^2_{ac}\;[\mathrm{GeV}^2/c^4]$")
ax.set_ylabel(r"$\varepsilon(m^2_{ac})$")
ax.set_title("True 1-D efficiency")
ax.set_ylim(-0.05, 1.0)
plt.tight_layout()
plt.show()

### Simulator

The simulator generates events with accept-reject using the efficiency as
the acceptance probability.  It takes a 3-element parameter vector and
returns `n_events` reconstructed Dalitz-plot points $(m^2_{ab}, m^2_{bc})$.

In [ ]:
def simulate_1d(parameters, n_events=2_000, chunk_size=20_000, rng=None):
    """
    Accept-reject simulator for the 1-D quadratic efficiency.

    Parameters
    ----------
    parameters : array-like, shape (3,)
        [p0, p1, p2] of the quadratic efficiency in m2ac.
    n_events : int
        Number of accepted events to return.
    chunk_size : int
        Candidate batch size per iteration.
    rng : np.random.Generator, optional
        Random number generator (for reproducibility).

    Returns
    -------
    torch.Tensor, shape (n_events, 2)
        Reconstructed (m2ab, m2bc) events.
    """
    if rng is None:
        rng = np.random.default_rng()
    p0, p1, p2 = [float(x) for x in np.asarray(parameters).flatten()]

    accepted = []
    n_kept = 0
    while n_kept < n_events:
        # Candidates: uniform in phase space + auxiliary U[0,1] for accept-reject
        candidates = phsp.uniform_sample(chunk_size, maximum=1.0)
        m2ac = phsp.m2ac(candidates)
        acc_prob = efficiency_1d(m2ac, p0, p1, p2)
        keep = candidates[:, 2] < acc_prob
        kept = candidates[keep, :2]
        if kept.shape[0] > 0:
            accepted.append(kept)
            n_kept += kept.shape[0]

    sample = np.concatenate(accepted, axis=0)[:n_events]
    return torch.as_tensor(sample, dtype=torch.float32)

In [ ]:
# Generate pseudo-data with the true parameters
np.random.seed(0)
N_DATA = 20_000
obs_data = simulate_1d(TRUE_PARAMS, n_events=N_DATA)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: Dalitz plot
plot_dalitz(phsp.m2ab(obs_data), phsp.m2bc(obs_data),
            ax=axes[0], bins=50)
axes[0].set_xlabel(r"$m^2_{ab}$"); axes[0].set_ylabel(r"$m^2_{bc}$")
axes[0].set_title("Reconstructed Dalitz plot (observed data)")

# Right: marginal in m²_ac
m2ac_obs = phsp.m2ac(obs_data)
axes[1].hist(m2ac_obs.numpy(), bins=40, density=True,
             histtype='stepfilled', alpha=0.5, color='steelblue', label='rec sample')
# Overlay: what the flat phase space looks like
gen_sample = phsp.uniform_sample(100_000)
axes[1].hist(phsp.m2ac(gen_sample), bins=40, density=True,
             histtype='step', color='gray', lw=1.5, label='flat MC')
axes[1].set_xlabel(r"$m^2_{ac}$"); axes[1].set_ylabel("Density")
axes[1].set_title(r"Marginal in $m^2_{ac}$")
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Part 2 — Neural Posterior Estimation (NPE)

### 2.1 Summary statistics

SBI requires a **fixed-dimensional** observation vector.  We compress the
variable-length Dalitz sample into a histogram of $m^2_{ac}$ (normalised to
a probability density).  Since the efficiency model depends only on $m^2_{ac}$,
this is a sufficient statistic for the parameters $(p_0, p_1, p_2)$.

To cancel phase-space shape effects, we compute the **efficiency ratio**
$\hat{\varepsilon}_i = n^{\rm rec}_i / n^{\rm gen}_i$ in each bin $i$, 
which is directly proportional to the true efficiency.  Flattening this
ratio vector gives our summary statistic.

In [ ]:
# Pre-compute the generator-level (flat phase-space) histogram once
N_BINS_1D = 20
BIN_EDGES_1D = np.linspace(phsp.minac, phsp.maxac, N_BINS_1D + 1)
BIN_CENTRES_1D = 0.5 * (BIN_EDGES_1D[:-1] + BIN_EDGES_1D[1:])

# Large flat MC for the denominator (fixed once and reused)
N_GEN = 500_000
flat_mc = phsp.uniform_sample(N_GEN)
h_gen, _ = np.histogram(phsp.m2ac(flat_mc), bins=BIN_EDGES_1D)
h_gen = h_gen.astype(float)


def summary_stats_1d(sample):
    """
    Compute 1-D summary statistics: efficiency ratio histogram.

    Parameters
    ----------
    sample : torch.Tensor or np.ndarray, shape (N, 2)
        (m2ab, m2bc) events.

    Returns
    -------
    torch.Tensor, shape (N_BINS_1D,)
    """
    if torch.is_tensor(sample):
        sample = sample.numpy()
    m2ac = phsp.m2ac(sample)
    h_rec, _ = np.histogram(m2ac, bins=BIN_EDGES_1D)
    # Efficiency ratio (scale so that ~1 at the most efficient bin)
    ratio = np.where(h_gen > 0, h_rec.astype(float) / h_gen, 0.0)
    ratio /= (ratio.max() + 1e-8)   # normalise to [0, 1]
    return torch.tensor(ratio, dtype=torch.float32)


# Observed summary statistics
x_obs_1d = summary_stats_1d(obs_data)
print(f"Summary stat dimension: {x_obs_1d.shape}")
print(f"Summary stat (obs): {x_obs_1d.numpy().round(3)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.step(BIN_CENTRES_1D, x_obs_1d.numpy(), where='mid', lw=1.8,
        label='observed summary stat', color='steelblue')
ax.plot(BIN_CENTRES_1D,
        efficiency_1d(BIN_CENTRES_1D, *TRUE_PARAMS)
        / efficiency_1d(BIN_CENTRES_1D, *TRUE_PARAMS).max(),
        'r--', lw=1.5, label='true eff (normalised)')
ax.set_xlabel(r"$m^2_{ac}$"); ax.set_ylabel("Normalised efficiency ratio")
ax.set_title("Summary statistics vs true efficiency")
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Prior

We place a uniform (box) prior over each parameter.  The ranges are chosen
to be physically reasonable:
- $p_0 \in [0.05, 0.95]$: efficiency at low $m^2_{ac}$.
- $p_1 \in [-0.5, 0.5]$: linear slope.
- $p_2 \in [-0.20, 0.05]$: quadratic curvature (mostly negative = concave).

In [ ]:
from sbi.utils import BoxUniform

prior_1d = BoxUniform(
    low  = torch.tensor([0.05, -0.5, -0.20]),
    high = torch.tensor([0.95,  0.5,  0.05]),
)

# Quick sanity check: does the true point lie inside the prior?
sample_check = prior_1d.sample((1,))
print(f"Prior sample shape : {sample_check.shape}")
print(f"True params        : {TRUE_PARAMS}")
print(f"Within prior range : p0={0.05 <= TRUE_PARAMS[0] <= 0.95}, "
      f"p1={-0.5 <= TRUE_PARAMS[1] <= 0.5}, "
      f"p2={-0.20 <= TRUE_PARAMS[2] <= 0.05}")

### 2.3 Training the neural posterior estimator

We use the `SNPE` class with an **NPE-C** (SNPE-C / APT) backbone.
The training loop:
1. Draw $\theta \sim \mathrm{Prior}$.
2. Run `simulate_1d(θ)` to get a Dalitz sample.
3. Compress to summary statistics $x = f(\mathrm{sample})$.
4. Feed $(\theta, x)$ pairs to the normalising-flow posterior estimator.

> **Note**: We use `n_simulations=2000` for speed here.  In a real analysis
> you would use 10 000–50 000 simulations.

In [ ]:
from sbi.inference import NPE_C
from sbi.utils.user_input_checks import check_sbi_inputs, process_prior

prior_1d_processed, num_parameters, prior_returns_numpy = process_prior(prior_1d)
inferrer_1d = NPE_C(prior=prior_1d_processed)

In [ ]:
%%time
N_SIM = 3_000   # number of training simulations
N_EVT = 3_000   # events per simulation (larger = sharper summary stats)

theta_train = prior_1d.sample((N_SIM,))
x_train = []

for i, theta in enumerate(theta_train):
    sim_sample = simulate_1d(theta, n_events=N_EVT)
    x_train.append(summary_stats_1d(sim_sample))
    if (i + 1) % 500 == 0:
        print(f"  Simulated {i+1}/{N_SIM}")

x_train = torch.stack(x_train)
print(f"theta_train: {theta_train.shape}, x_train: {x_train.shape}")

In [ ]:
%%time
inferrer_1d.append_simulations(theta_train, x_train)
density_estimator_1d = inferrer_1d.train(
    training_batch_size=128,
    learning_rate=5e-4,
    max_num_epochs=50,
    show_train_summary=True,
)

### 2.4 Posterior inference

We condition the trained estimator on the **observed summary statistics** and
draw samples from the approximate posterior $q(\theta \mid x_{\rm obs})$.

In [ ]:
posterior_1d = inferrer_1d.build_posterior(density_estimator_1d)

N_POST = 10_000
posterior_samples_1d = posterior_1d.sample(
    (N_POST,),
    x=x_obs_1d,
    show_progress_bars=True,
)
print(f"Posterior samples: {posterior_samples_1d.shape}")

---
## Part 3 — Diagnostics

### 3.1 Posterior corner plot

In [ ]:
try:
    import corner
    HAS_CORNER = True
except ImportError:
    HAS_CORNER = False

param_labels = [r'$p_0$', r'$p_1$', r'$p_2$']
post_np = posterior_samples_1d.numpy()

if HAS_CORNER:
    fig = corner.corner(
        post_np,
        labels=param_labels,
        truths=TRUE_PARAMS,
        truth_color='tomato',
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_kwargs={'fontsize': 10},
    )
    fig.suptitle("Posterior (1-D efficiency)", y=1.01)
else:
    # Minimal corner-like plot using matplotlib
    fig, axes = plt.subplots(3, 3, figsize=(7, 7))
    for i in range(3):
        for j in range(3):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
            elif i == j:
                ax.hist(post_np[:, i], bins=40, color='steelblue', density=True)
                ax.axvline(TRUE_PARAMS[i], color='tomato', lw=2)
                ax.set_xlabel(param_labels[i])
            else:
                ax.scatter(post_np[:, j], post_np[:, i], s=1, alpha=0.1,
                           color='steelblue')
                ax.scatter(TRUE_PARAMS[j], TRUE_PARAMS[i], s=60, color='tomato',
                           zorder=5, marker='*')
                ax.set_xlabel(param_labels[j])
                ax.set_ylabel(param_labels[i])
    plt.suptitle("Posterior (1-D efficiency)", y=1.01)
    plt.tight_layout()

plt.show()

In [ ]:
# Posterior median and 68% credible intervals
print("\nPosterior summary (median ± 68% CI vs truth):")
print(f"{'Param':>8} {'Truth':>10} {'Median':>10} {'−1σ':>10} {'+1σ':>10}")
print("-" * 52)
for k, (label, true) in enumerate(zip(param_labels, TRUE_PARAMS)):
    vals = post_np[:, k]
    lo, med, hi = np.percentile(vals, [16, 50, 84])
    print(f"{label:>8} {true:>10.5f} {med:>10.5f} {lo-med:>10.5f} {hi-med:>+10.5f}")

### 3.2 Posterior predictive check (PPC)

We draw a handful of parameter samples from the posterior and overlay the
predicted efficiency curves on top of the inferred efficiency from the data
(ratio rec/gen).  The true efficiency curve should sit within the posterior
predictive band.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Left: efficiency curves ---
ax = axes[0]
x_plot = np.linspace(phsp.minac, phsp.maxac, 300)

# 100 posterior samples
idx = np.random.choice(len(post_np), 200, replace=False)
for p in post_np[idx]:
    ax.plot(x_plot, efficiency_1d(x_plot, *p), color='steelblue', alpha=0.05, lw=0.8)

# Posterior median
p_med = np.median(post_np, axis=0)
ax.plot(x_plot, efficiency_1d(x_plot, *p_med), color='navy', lw=2,
        label='Posterior median')
# True
ax.plot(x_plot, efficiency_1d(x_plot, *TRUE_PARAMS), 'r--', lw=2, label='Truth')

# Binned data-driven efficiency (rec/gen)
h_rec_obs, _ = np.histogram(phsp.m2ac(obs_data).numpy(), bins=BIN_EDGES_1D)
h_gen_norm = h_gen * (N_DATA / N_GEN)   # scale gen to match rec counts
eff_data = np.where(h_gen_norm > 0, h_rec_obs / h_gen_norm, 0.0)
eff_err  = np.where(h_gen_norm > 0,
                    np.sqrt(h_rec_obs) / h_gen_norm, 0.0)
ax.errorbar(BIN_CENTRES_1D, eff_data, yerr=eff_err,
            fmt='ko', ms=4, zorder=5, label='Data (rec/gen)')

ax.set_xlabel(r"$m^2_{ac}$"); ax.set_ylabel(r"$\varepsilon$")
ax.set_title("Posterior predictive: efficiency")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)

# --- Right: Dalitz-plot residuals ---
ax2 = axes[1]

# Generate a predictive sample from the posterior median
pred_sample = simulate_1d(p_med, n_events=N_DATA)

# 1D projection on m2ab
bins_ab = 40
rng_ab = (phsp.minab, phsp.maxab)
h_obs,  edges = np.histogram(phsp.m2ab(obs_data).numpy(),  bins=bins_ab, range=rng_ab)
h_pred, _     = np.histogram(phsp.m2ab(pred_sample).numpy(), bins=bins_ab, range=rng_ab)
centres_ab = 0.5 * (edges[:-1] + edges[1:])
ax2.step(centres_ab, h_obs,  label='Observed', where='mid', color='k')
ax2.step(centres_ab, h_pred, label='Posterior median prediction', where='mid',
         color='steelblue', lw=1.5)
ax2.set_xlabel(r"$m^2_{ab}$"); ax2.set_ylabel("Events / bin")
ax2.set_title(r"PPC: $m^2_{ab}$ projection")
ax2.legend()

plt.tight_layout()
plt.show()

---
## Part 4 — Extension: 2-D efficiency on the Dalitz plot

A real LHCb detector efficiency depends on the full 2-D phase space
$(m^2_{ab}, m^2_{bc})$ — not just one projection.  We model it as a sigmoid
of a bilinear function:

$$\varepsilon(m^2_{ab}, m^2_{bc};\, \mathbf{\alpha})
= \sigma\!\left(\alpha_0 + \alpha_1\tilde{x} + \alpha_2\tilde{y}
                 + \alpha_3\tilde{x}^2 + \alpha_4\tilde{y}^2 + \alpha_5\tilde{x}\tilde{y}\right)$$

where $\tilde{x}$, $\tilde{y}$ are linearly normalised $m^2_{ab}$, $m^2_{bc}$
to $[-1, 1]$, and $\sigma$ is the logistic function.  This guarantees
$\varepsilon \in (0, 1)$ by construction.

The summary statistics are the entries of a **flattened 2-D efficiency-ratio
histogram** on the Dalitz plot.

In [ ]:
def _normalise_ab(m2ab):
    """Linear map m2ab -> [-1, 1]."""
    c = 0.5 * (phsp.maxab + phsp.minab)
    h = 0.5 * (phsp.maxab - phsp.minab)
    return (m2ab - c) / h


def _normalise_bc(m2bc):
    """Linear map m2bc -> [-1, 1]."""
    c = 0.5 * (phsp.maxbc + phsp.minbc)
    h = 0.5 * (phsp.maxbc - phsp.minbc)
    return (m2bc - c) / h


def efficiency_2d(m2ab, m2bc, alpha):
    """
    2-D efficiency via logistic of a quadratic form.

    Parameters
    ----------
    m2ab, m2bc : array-like
    alpha : array-like, shape (6,)
        [a0, a1, a2, a3, a4, a5] coefficients.

    Returns
    -------
    array-like in (0, 1)
    """
    x = _normalise_ab(m2ab)
    y = _normalise_bc(m2bc)
    a0, a1, a2, a3, a4, a5 = alpha
    logit = a0 + a1 * x + a2 * y + a3 * x**2 + a4 * y**2 + a5 * x * y
    return 1.0 / (1.0 + np.exp(-logit))


# True 2-D parameters: a moderate efficiency gradient + a cross-term
# mimicking a detector with lower acceptance in the high-m2ab, low-m2bc corner
ALPHA_TRUE = np.array([-0.3,   # overall offset (avg eff ~ sigmoid(-0.3) ≈ 0.43)
                        -1.0,   # decrease with m2ab
                         0.8,   # increase with m2bc
                        -0.4,   # concave in m2ab
                        -0.3,   # concave in m2bc
                         0.5])  # cross-term

# Visualise true 2-D efficiency
grid_ab = np.linspace(phsp.minab, phsp.maxab, 100)
grid_bc = np.linspace(phsp.minbc, phsp.maxbc, 100)
M2AB, M2BC = np.meshgrid(grid_ab, grid_bc)
EFF_2D_TRUE = efficiency_2d(M2AB, M2BC, ALPHA_TRUE)

# Mask phase-space boundary (approximate with m2ac > 0)
m2ac_grid = phsp.msqsum - M2AB - M2BC
in_phsp = (m2ac_grid > phsp.minac) & (m2ac_grid < phsp.maxac)
EFF_2D_TRUE_MASKED = np.where(in_phsp, EFF_2D_TRUE, np.nan)

fig, ax = plt.subplots(figsize=(6, 4.5))
cm = ax.pcolormesh(grid_ab, grid_bc, EFF_2D_TRUE_MASKED,
                   cmap='RdYlGn', vmin=0, vmax=1)
fig.colorbar(cm, ax=ax, label=r'$\varepsilon(m^2_{ab}, m^2_{bc})$')
ax.set_xlabel(r"$m^2_{ab}$"); ax.set_ylabel(r"$m^2_{bc}$")
ax.set_title("True 2-D efficiency")
plt.tight_layout()
plt.show()

print(f"Mean efficiency (in phsp): {np.nanmean(EFF_2D_TRUE_MASKED):.3f}")

In [ ]:
def simulate_2d(alpha, n_events=2_000, chunk_size=30_000):
    """
    Accept-reject simulator for the 2-D efficiency.

    Parameters
    ----------
    alpha : array-like, shape (6,)
    n_events : int
    chunk_size : int

    Returns
    -------
    torch.Tensor, shape (n_events, 2)   [m2ab, m2bc]
    """
    alpha = np.asarray(alpha).flatten()
    accepted, n_kept = [], 0
    while n_kept < n_events:
        candidates = phsp.uniform_sample(chunk_size, maximum=1.0)
        m2ab = phsp.m2ab(candidates)
        m2bc = phsp.m2bc(candidates)
        acc_prob = efficiency_2d(m2ab, m2bc, alpha)
        keep = candidates[:, 2] < acc_prob
        kept = candidates[keep, :2]
        if kept.shape[0] > 0:
            accepted.append(kept)
            n_kept += kept.shape[0]
    sample = np.concatenate(accepted)[:n_events]
    return torch.as_tensor(sample, dtype=torch.float32)


# Generate 2-D pseudo-data
np.random.seed(1)
obs_data_2d = simulate_2d(ALPHA_TRUE, n_events=N_DATA)

fig, ax = plt.subplots(figsize=(5, 4))
plot_dalitz(phsp.m2ab(obs_data_2d), phsp.m2bc(obs_data_2d), ax=ax, bins=50)
ax.set_xlabel(r"$m^2_{ab}$"); ax.set_ylabel(r"$m^2_{bc}$")
ax.set_title("Observed Dalitz plot (2-D efficiency)")
plt.tight_layout()
plt.show()

### 4.1 2-D summary statistics

We use a flattened 2-D efficiency-ratio histogram on the $(m^2_{ab}, m^2_{bc})$
plane (10 × 10 bins, giving 100-dimensional summary statistics).

In [ ]:
N_BINS_2D = 10
BIN_EDGES_AB = np.linspace(phsp.minab, phsp.maxab, N_BINS_2D + 1)
BIN_EDGES_BC = np.linspace(phsp.minbc, phsp.maxbc, N_BINS_2D + 1)

# 2-D flat MC histogram (fixed normalisation reference)
flat_mc_2d = phsp.uniform_sample(N_GEN)
h_gen_2d, _, _ = np.histogram2d(
    phsp.m2ab(flat_mc_2d), phsp.m2bc(flat_mc_2d),
    bins=[BIN_EDGES_AB, BIN_EDGES_BC]
)


def summary_stats_2d(sample):
    """
    Compute 2-D summary statistics: flattened efficiency-ratio histogram.

    Returns
    -------
    torch.Tensor, shape (N_BINS_2D * N_BINS_2D,)
    """
    if torch.is_tensor(sample):
        sample = sample.numpy()
    h_rec, _, _ = np.histogram2d(
        phsp.m2ab(sample), phsp.m2bc(sample),
        bins=[BIN_EDGES_AB, BIN_EDGES_BC]
    )
    ratio = np.where(h_gen_2d > 0, h_rec / (h_gen_2d + 1e-8), 0.0)
    ratio /= (ratio.max() + 1e-8)
    return torch.tensor(ratio.flatten(), dtype=torch.float32)


x_obs_2d = summary_stats_2d(obs_data_2d)
print(f"2-D summary stat dimension: {x_obs_2d.shape}")

# Visualise observed 2-D summary stats
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.pcolormesh(
    BIN_EDGES_AB, BIN_EDGES_BC,
    x_obs_2d.numpy().reshape(N_BINS_2D, N_BINS_2D).T,
    cmap='RdYlGn', vmin=0, vmax=1
)
fig.colorbar(im, ax=ax, label='Normalised ratio')
ax.set_xlabel(r"$m^2_{ab}$"); ax.set_ylabel(r"$m^2_{bc}$")
ax.set_title("Observed 2-D summary statistics")
plt.tight_layout()
plt.show()

In [ ]:
from sbi.utils import BoxUniform

prior_2d = BoxUniform(
    low  = torch.tensor([-2.0, -2.0, -2.0, -1.5, -1.5, -1.5]),
    high = torch.tensor([ 2.0,  2.0,  2.0,  1.5,  1.5,  1.5]),
)

print(f"True alpha  : {ALPHA_TRUE}")
print(f"Within prior: {np.all(np.abs(ALPHA_TRUE) < np.array([2, 2, 2, 1.5, 1.5, 1.5]))}")

In [ ]:
%%time
from sbi.inference import NPE_C
from sbi.utils.user_input_checks import process_prior

prior_2d_proc, _, _ = process_prior(prior_2d)
inferrer_2d = NPE_C(prior=prior_2d_proc)

N_SIM_2D = 3_000
N_EVT_2D = 3_000

theta_2d = prior_2d.sample((N_SIM_2D,))
x_2d = []

for i, alpha in enumerate(theta_2d):
    sim = simulate_2d(alpha, n_events=N_EVT_2D)
    x_2d.append(summary_stats_2d(sim))
    if (i + 1) % 500 == 0:
        print(f"  Simulated {i+1}/{N_SIM_2D}")

x_2d = torch.stack(x_2d)
print(f"theta_2d: {theta_2d.shape}, x_2d: {x_2d.shape}")

inferrer_2d.append_simulations(theta_2d, x_2d)
density_estimator_2d = inferrer_2d.train(
    training_batch_size=128,
    learning_rate=5e-4,
    max_num_epochs=50,
    show_train_summary=True,
)

In [ ]:
posterior_2d = inferrer_2d.build_posterior(density_estimator_2d)
post_samples_2d = posterior_2d.sample(
    (N_POST,),
    x=x_obs_2d,
    show_progress_bars=True,
)
post_2d_np = post_samples_2d.numpy()

alpha_labels = [r'$\alpha_0$', r'$\alpha_1$', r'$\alpha_2$',
                r'$\alpha_3$', r'$\alpha_4$', r'$\alpha_5$']
print("\nPosterior summary (median ± 68% CI vs truth):")
print(f"{'Param':>8} {'Truth':>8} {'Median':>8} {'−1σ':>8} {'+1σ':>8}")
print("-" * 48)
for k, (lbl, true) in enumerate(zip(alpha_labels, ALPHA_TRUE)):
    vals = post_2d_np[:, k]
    lo, med, hi = np.percentile(vals, [16, 50, 84])
    print(f"{lbl:>8} {true:>8.3f} {med:>8.3f} {lo-med:>8.3f} {hi-med:>+8.3f}")

### 4.2 Inferred 2-D efficiency map

We compare the **true** efficiency map with the one reconstructed from the
**posterior median** and the **binned rec/gen ratio** from data.

In [ ]:
alpha_median = np.median(post_2d_np, axis=0)

# Predicted efficiency on a fine grid
EFF_2D_PRED = efficiency_2d(M2AB, M2BC, alpha_median)
EFF_2D_PRED_MASKED = np.where(in_phsp, EFF_2D_PRED, np.nan)

# Data-driven rec/gen ratio (coarse, 20x20 bins)
N_COARSE = 20
edges_ab_c = np.linspace(phsp.minab, phsp.maxab, N_COARSE + 1)
edges_bc_c = np.linspace(phsp.minbc, phsp.maxbc, N_COARSE + 1)
h_rec_2d, _, _ = np.histogram2d(
    phsp.m2ab(obs_data_2d).numpy(), phsp.m2bc(obs_data_2d).numpy(),
    bins=[edges_ab_c, edges_bc_c]
)
h_gen_c, _, _ = np.histogram2d(
    phsp.m2ab(flat_mc_2d), phsp.m2bc(flat_mc_2d),
    bins=[edges_ab_c, edges_bc_c]
)
eff_ratio_data = np.where(h_gen_c > 0,
                          h_rec_2d / (h_gen_c * N_DATA / N_GEN), np.nan)

# --- Three-panel comparison ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
kw = dict(cmap='RdYlGn', vmin=0, vmax=1)

# True
cm0 = axes[0].pcolormesh(grid_ab, grid_bc, EFF_2D_TRUE_MASKED, **kw)
axes[0].set_title("True efficiency")
fig.colorbar(cm0, ax=axes[0])

# Posterior median prediction
cm1 = axes[1].pcolormesh(grid_ab, grid_bc, EFF_2D_PRED_MASKED, **kw)
axes[1].set_title("Posterior median (SBI)")
fig.colorbar(cm1, ax=axes[1])

# Data rec/gen ratio
cen_ab = 0.5 * (edges_ab_c[:-1] + edges_ab_c[1:])
cen_bc = 0.5 * (edges_bc_c[:-1] + edges_bc_c[1:])
cm2 = axes[2].pcolormesh(edges_ab_c, edges_bc_c, eff_ratio_data.T, **kw)
axes[2].set_title("Data rec/gen ratio")
fig.colorbar(cm2, ax=axes[2], label=r'$\varepsilon$')

for ax in axes:
    ax.set_xlabel(r"$m^2_{ab}$"); ax.set_ylabel(r"$m^2_{bc}$")

plt.suptitle("2-D efficiency comparison", y=1.02)
plt.tight_layout()
plt.show()

### 4.3 Residuals and uncertainty

In [ ]:
# Pull: (pred - true) / std over posterior samples
eff_post_samples = np.stack([
    efficiency_2d(M2AB, M2BC, alpha)
    for alpha in post_2d_np[:500]
], axis=0)   # (500, 100, 100)

eff_post_std  = np.std(eff_post_samples, axis=0)
eff_post_mean = np.mean(eff_post_samples, axis=0)
pull = np.where(in_phsp & (eff_post_std > 0),
                (eff_post_mean - EFF_2D_TRUE) / eff_post_std, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Posterior uncertainty
cm0 = axes[0].pcolormesh(grid_ab, grid_bc,
                          np.where(in_phsp, eff_post_std, np.nan),
                          cmap='Blues', vmin=0)
fig.colorbar(cm0, ax=axes[0], label=r'$\sigma(\varepsilon)$')
axes[0].set_title("Posterior uncertainty on efficiency")

# Pull
cm1 = axes[1].pcolormesh(grid_ab, grid_bc, pull,
                          cmap='RdBu_r', vmin=-3, vmax=3)
fig.colorbar(cm1, ax=axes[1], label='Pull')
axes[1].set_title(r"Pull = $(\bar{\varepsilon} - \varepsilon_{\rm true})/\sigma$")

for ax in axes:
    ax.set_xlabel(r"$m^2_{ab}$"); ax.set_ylabel(r"$m^2_{bc}$")
plt.tight_layout()
plt.show()

# Summary
pull_flat = pull[in_phsp & ~np.isnan(pull)]
print(f"Pull statistics (in phase space):")
print(f"  mean  = {pull_flat.mean():.3f}")
print(f"  std   = {pull_flat.std():.3f}  (should be ~1 for a well-calibrated posterior)")
print(f"  |>2σ| = {(np.abs(pull_flat) > 2).mean()*100:.1f}%  (expect ~5%)")

---
## Summary

| | 1-D efficiency | 2-D efficiency |
|---|---|---|
| **Parameters** | 3 (quadratic in $m^2_{ac}$) | 6 (bilinear form, sigmoid) |
| **Summary stats** | 20-bin $m^2_{ac}$ histogram | 10×10 2-D ratio histogram |
| **Simulations** | 3 000 | 3 000 |
| **Events/simulation** | 3 000 | 3 000 |
| **Method** | NPE (SNPE-C) | NPE (SNPE-C) |

### Key points

- **SBI avoids explicit likelihood construction**: the efficiency map does not
  need to be analytically tractable — we only need a forward simulator.
- **Summary statistics matter**: using the 1-D or 2-D efficiency-ratio
  histogram as the observation compresses the sample faithfully while being
  well-matched to the model structure.  More general options include
  normalising flows or neural compression trained end-to-end.
- **Calibration**: the pull plots verify that the posterior uncertainty is
  not over- or under-confident.
- **Next steps for a real LHCb analysis**:
  1. Replace the parametric efficiency with a non-parametric model
     (e.g. normalising-flow amplitude or neural-network efficiency map).
  2. Use **Sequential SBI** (active learning) to focus simulations near the
     posterior mode for efficiency.
  3. Add the **signal amplitude** to the Dalitz plot model so that
     efficiency and amplitude parameters are inferred jointly.